In [ ]:
import os
# Train on a single GPU — HF Trainer otherwise spreads params across visible GPUs
# which breaks TRL's single-source NCCL weight-sync to the vLLM server
# (pynccl expects all params on the same device as the communicator).
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["WANDB_DISABLED"] = "true"
# DeepSpeed (pulled in by accelerate) probes torch.utils.cpp_extension.CUDA_HOME at import time.
# No nvcc in default PATH on this node — point at the HPC SDK CUDA toolkit.
os.environ["CUDA_HOME"] = "/opt/nvidia/hpc_sdk/Linux_aarch64/24.11/cuda/12.6"

import torch, trl, inspect_ai, transformers
print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}")
print(f"trl: {trl.__version__} | inspect-ai: {inspect_ai.__version__} | transformers: {transformers.__version__}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.mem_get_info(i)[0]//1024**3}GB free")

In [ ]:
import httpx
resp = httpx.get("http://localhost:8000/health/", timeout=5)
print(f"vLLM health: {resp.json()}")

# Quick generation test
resp = httpx.post("http://localhost:8000/chat/", json={
    "messages": [[{"role": "user", "content": "What is 2+2?"}]],
    "max_tokens": 50, "n": 1,
}, timeout=30)
data = resp.json()
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")
print(f"Test generation: {tok.decode(data['completion_ids'][0], skip_special_tokens=True)[:200]}")

In [ ]:
from inspect_rl.example.gsm8k_v2 import get_task
from inspect_rl.trainer import inspect_rl_train
from peft import LoraConfig, TaskType
from trl import GRPOConfig

MODEL = "google/gemma-3-270m-it"

# task = get_task()
# print(f"Task: {len(list(task.dataset))} samples")

# for s in task.scorer:
#     print(f"  scorer: __qualname__={getattr(s, '__qualname__', '?')}")

# grpo_config = GRPOConfig(
#     output_dir="outputs/debug_gemma3",
#     max_steps=15,
#     per_device_train_batch_size=4,
#     num_generations=4,
#     max_completion_length=1024,
#     temperature=1.0,
#     learning_rate=1e-5,
#     warmup_steps=2,
#     bf16=True,
#     fp16=False,
#     report_to="none",
#     logging_steps=1,
#     use_vllm=True,
#     vllm_mode="server",
#     vllm_server_host="localhost",
#     vllm_server_port=8000,
#     reward_weights=[2.0, 3.0],
# )

peft_config = LoraConfig(
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM,
    lora_alpha=32,
    lora_dropout=0.0,
)

# print(f"\nConfig ready — MODEL={MODEL}, batch_size=4, num_generations=4, max_steps=5, no wandb")

In [ ]:
import importlib
import inspect_rl.rollout, inspect_rl.trainer
importlib.reload(inspect_rl.rollout)
importlib.reload(inspect_rl.trainer)
from inspect_rl.trainer import inspect_rl_train

# Clear any stale NCCL weight-update group on the vLLM server.
# Needed if a previous trainer in this kernel/session died without running atexit cleanup.
import httpx
try:
    httpx.post("http://localhost:8000/close_communicator/", json={}, timeout=10)
except Exception as e:
    print(f"close_communicator (ignorable if first run): {e}")

# Gemma-3 requires token_type_ids at training time (for text/image bidirectional masking).
# TRL's GRPOTrainer doesn't pass one since we're text-only — inject zeros so the text-only path works.
import torch
from transformers.models.gemma3 import modeling_gemma3
if not getattr(modeling_gemma3.Gemma3Model.forward, "_tti_patched", False):
    _orig_gemma3_forward = modeling_gemma3.Gemma3Model.forward
    def _gemma3_forward_tti(self, *args, **kwargs):
        if kwargs.get("token_type_ids") is None and kwargs.get("input_ids") is not None:
            kwargs["token_type_ids"] = torch.zeros_like(kwargs["input_ids"])
        return _orig_gemma3_forward(self, *args, **kwargs)
    _gemma3_forward_tti._tti_patched = True
    modeling_gemma3.Gemma3Model.forward = _gemma3_forward_tti

# inspect_rl_train(
#     task=task,
#     model=MODEL,
#     grpo_config=grpo_config,
#     vllm_base_url="http://localhost:8000",
#     peft_config=peft_config,
#     dataset_limit=100,
# )

## TLDR smoke test

Run `tldr_v2.train` for 5 steps to confirm the v2 pipeline works on a second task. Reward is simple length-penalty + first-person bonus — easy to move the needle on quickly.

In [ ]:
import importlib
import inspect_rl.example.tldr_v2 as tldr_v2_mod
importlib.reload(tldr_v2_mod)
from inspect_rl.example.tldr_v2 import get_task as get_tldr_task
from trl import GRPOConfig

tldr_task = get_tldr_task()
print(f"TLDR task: {len(list(tldr_task.dataset))} samples")
for s in tldr_task.scorer:
    print(f"  scorer: {getattr(s, '__qualname__', '?')}")

# Match v1 defaults: no LoRA, full fine-tuning, default LR/batch/generations.
# Only v2-specific additions are the vLLM server config.
tldr_grpo_config = GRPOConfig(
    output_dir="outputs/debug_tldr_v2",
    max_steps=50,
    bf16=True,
    fp16=False,
    report_to="none",
    save_steps=50,
    logging_steps=1,
    # v2 architecture: vLLM server for generation
    use_vllm=True,
    vllm_mode="server",
    vllm_server_host="localhost",
    vllm_server_port=8000,
    reward_weights=[1.0],
)

print(f"TLDR config ready — full FT (no LoRA), {tldr_grpo_config.max_steps} steps")
print(f"  batch_size={tldr_grpo_config.per_device_train_batch_size}, "
      f"num_generations={tldr_grpo_config.num_generations}, "
      f"lr={tldr_grpo_config.learning_rate}, "
      f"max_completion_length={tldr_grpo_config.max_completion_length}")

In [ ]:
# Close any stale NCCL weight-update group before creating a fresh trainer.
try:
    httpx.post("http://localhost:8000/close_communicator/", json={}, timeout=10)
except Exception as e:
    print(f"close_communicator (ignorable if first run): {e}")

# Full fine-tuning (no peft_config) — matches v1 tldr.py behavior.
trainer = inspect_rl_train(
    task=tldr_task,
    model=MODEL,
    grpo_config=tldr_grpo_config,
    vllm_base_url="http://localhost:8000",
    peft_config=None,
    dataset_limit=1000,
)

save_path = "outputs/tldr_v2_final"
trainer.save_model(save_path)
print(f"Model saved to {save_path}")